## 1. Data Scraping

This section scrapes live property listings from [NigeriaPropertyCentre](https://nigeriapropertycentre.com) 
using `requests` to fetch raw HTML and `BeautifulSoup` to parse and extract structured data.

We target Lagos properties listed **for sale**, scraping across **500 pages** — approximately 
10,000 listings. Each listing is parsed for the following fields:

| Field | Description |
|---|---|
| `title` | Property name/type as listed |
| `price` | Asking price in Naira |
| `location` | Area/neighborhood in Lagos |
| `bedrooms` | Number of bedrooms |
| `bathrooms` | Number of bathrooms |
| `toilets` | Number of toilets |
| `Packing Space` | Packing space exist or not |

### Scraping Strategy
- A custom function scrapes one page at a time and returns a list of records
- A loop iterates that function across 500 pages
- A `time.sleep()` delay is added between requests to avoid overloading the server
- Raw data is saved immediately to CSV after scraping completes

### Notebook Structure
- **1.1** Scraping Configuration
- **1.2** Single Page Scraper Function
- **1.3** Multi-Page Scraper Loop
- **1.4** Save Raw Data to CSV

In [1]:
# importing Cell

import time
import pandas as pd
from bs4 import BeautifulSoup
import requests


print("libraries loaded successfully")

libraries loaded successfully


##### 1.1 Scraping Configuration

In [ ]:
num_pages = 500
base_url = "https://nigeriapropertycentre.com/for-sale?keywords=Lagos&page={}"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

records = {
  "Title": [],
  "Price": [],
  "Location": [],
  "Bedrooms": [],
  "Bathrooms": [],
  "Property_Type": [],
  "URL": [],
  "Toilets": [],
  "Parking_Space": []
}


##### 1.2 Single Page Scraper Function

In [ ]:
def scrape_page(page_num):
  url = base_url.format(page_num)
  page = requests.get(url, headers=headers)
  soup = BeautifulSoup(page.content, "html.parser")

  listings = soup.find_all("article")

  for listing in listings:
    try:
      # Title
      title_tag = listing.find("h3")
      titles = title_tag.get_text(strip=True) if title_tag else "N/A"

      #price
      price_tag = listing.find("span", class_="tabular-nums")
      prices = price_tag.get_text(strip=True) if price_tag else "N/A"

      #Property Type
      type_tag = listing.find("p", class_="text-primary")
      property_type = type_tag.get_text(strip=True) if type_tag else "N/A"

      #Property URL
      url_tag = listing.find("a", class_="absolute")
      prop_url = "https://nigeriapropertycentre.com" + url_tag["href"] if url_tag else "N/A"

      #location
      href = url_tag["href"]
      parts = href.split("/")
      if len(parts) > 5 and not parts[5][0].isdigit():
        location = f"{parts[5]}, {parts[4]}, {parts[3]}"
      else:
        location = f"{parts[4]}, {parts[3]}"

      # Beds, Baths, Toilets, Parking
      bedrooms = bathrooms = toilets = parking = "N/A"
      for span in listing.find_all("span", class_ = "inline-flex"):
        text = span.text.strip()
        if "Bed" in text:
          bedrooms = text.replace("Beds", "").replace("Bed", "").strip()
        elif "Toilet" in text:
          toilets = text.replace("Toilets", "").replace("Toilet", "").strip()
        elif "Bath" in text:
          bathrooms = text.replace("Baths", "").replace("Bath", "").strip()
        elif "Parking" in text:
          parking = text.replace("Parking", "").strip()

      records["Bedrooms"].append(bedrooms)
      records["Toilets"].append(toilets)
      records["Bathrooms"].append(bathrooms)
      records["Parking_Space"].append(parking)
      records["Title"].append(titles)
      records["Price"].append(prices)
      records["Property_Type"].append(property_type)
      records["URL"].append(prop_url)
      records["Location"].append(location)
    except Exception as e
      continue

##### 1.3 Multi-Page Scraper Loop

In [ ]:
for page in range(1, 500 + 1):
  try:
    scrape_page(page)
    print(f"Page {page} scraped successfully")
  except Exception as e:
    print(f"Error scraping page {page}: {e}")
  time.sleep(3) 
  if page % 50 == 0:
    print(f"--- Progress: {len(records["Title"])} listings collected so far ---")

#### 1.4 Save raw data to csv

In [ ]:
df = pd.DataFrame(records)
df.to_csv("../data/lagosIsland.csv", index = False)
df.head()